# 异常处理 + 文件 IO —— 进阶（练习）

## 1. 异常粒度 + else子句 + 主动raise

In [1]:
# 坏习惯：裸except，把所有异常都吞了（包括你没预料的bug）
def bad_parse(s):
    try:
        return int(s)
    except: # 裸except，连KeyboardInterrupt都抓——千万别这样
        return None
    
# 好习惯：只抓你预期的异常类型
def good_parse(s):
    try:
        return int(s)
    except ValueError: # 只抓“转换失败”这一种
        return None

else子句——try没出错时才执行：

In [2]:
def read_config(s):
    try:
        value = int(s)
    except ValueError:
        print("解析失败")
        return None
    else:
        # 只有try成功（没异常）才走这里
        print(f"解析成功：{value}")
        return value

try/except/else/finally四件套的分工：try放可能出错的、except处理错误、else放成功后才做的事、finally放无论如何都要做的事（如关闭资源）。把成功逻辑放else而不是try里，能让try块更小、异常来源更清晰。

主动raise——自己抛异常：

In [3]:
def set_age(age):
    if age < 0:
        raise ValueError(f"年龄不能为负：{age}")
    return age

# 调用方处理
try:
    set_age(-5)
except ValueError as e:
    print(f"捕获：{e}")

捕获：年龄不能为负：-5


# 2. 自定义异常 + 异常链

In [4]:
# 自定义异常：继承Exception,给你的业务一个专属错误类型
class DataValidationError(Exception):
    """数据校验失败时抛出"""
    pass

def validate_order(order):
    if "total" not in order:
        raise DataValidationError("订单缺少total字段")
    if order["total"] < 0:
        raise DataValidationError(f"total不能为负：{order['total']}")
    return True

try:
    validate_order({"product": "Laptop"})
except DataValidationError as e:
    print(f"校验错误：{e}")

校验错误：订单缺少total字段


异常链raise...from——把底层异常包装成业务异常，但保留原始原因：

In [6]:
def parse_price(s):
    try:
        return float(s)
    except ValueError as e:
        # 把底层的ValueError包装成业务异常，from e保留原始追溯
        raise DataValidationError(f"价格格式错误：{s}") from e

try:
    parse_price("abc")
except DataValidationError as e:
    print(f"业务错误：{e}")
    print(f"根本原因：{e.__cause__}") # 还能拿到原始异常

业务错误：价格格式错误：abc
根本原因：could not convert string to float: 'abc'


自定义异常的价值：调用方能用except DataValidationError精准抓你的业务错误，而不是和系统的ValueError混在一起。中大型项目里很常见。

## 3. logging替代print

Day 9题10你写坏数据就该跳过+记日志 —— 这就是logging干的事。它比print专业：有级别、有时间戳、能输出到文件。

In [8]:
import logging

# 配置一次（通常放程序开头）
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

def process_record(record):
    try:
        total = float(record["total"])
    except (KeyError, ValueError) as e:
        logging.warning(f"跳过坏数据 {record}：{e}") # 记下来，而不是默默丢
        return None
    return total

process_record({"total": "100"}) # 正常
process_record({"total": "abc"}) # 打印WARNING日志
process_record({"product": "X"}) # 缺total，打印WARNING


2026-05-28 13:36:49,500 [WARNING] 跳过坏数据 {'total': 'abc'}：could not convert string to float: 'abc'
2026-05-28 13:36:49,502 [WARNING] 跳过坏数据 {'product': 'X'}：'total'


日志级别（从低到高）：DEBUG < INFO < WARNING < ERROR < CRITICAL。设level=INFO就只显示INFO及以上。

数据管道的黄金法则：坏数据跳过，但一定要logging.warning记下来——否则数据出问题你完全不知道。print调试用，正经代码用logging。


## 4. pathlib + with深入（解决你的路径痛点）

还记得你被../data/sales.csv的相对路径折腾过吗？pathlib是现代Python处理路径的标准方式，跨平台（Windows反斜杠它自动处理）。

In [9]:
from pathlib import Path

# 构造路径——用/运算符拼接，不用手写斜杠
data_dir = Path("..")/"data"
csv_path = data_dir / "sales.csv"
print(csv_path) # ..\data\sales.csv (Windows) 或 ../data/sales.csv

# 实用方法
print(csv_path.exists()) # 文件存不存在——可以先检查再读
print(csv_path.name) # sales.csv
print(csv_path.suffix) # .csv
print(Path.cwd()) # 当前工作目录（等于 os.getcwd()）

..\data\sales.csv
True
sales.csv
.csv
c:\Users\69261\Desktop\data-skills-learning\week02


In [ ]:
# pathlib + with 读文件：先检查存在性，更优雅
def safe_read(path_str):
    path = Path(path_str)
    if not path.exists():
        logging.warning(f"文件不存在：{path}")
        return None
    with path.open("r", encoding="utf-8") as f: # Path对象直接.open()
        return f.read()

Windows + Git Bash下路径混乱（\ vs /）的问题， pathlib基本能消掉。以后写路径优先用它。

## 5. csv + json模块读写

In [11]:
import csv
from pathlib import Path

# 读CSV —— DictReader把每行读成dict
csv_path = Path("..") / "data" / "sales.csv"
with csv_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
print(f"读到{len(rows)}行")
print(rows[0])
# 注意：csv读出来的值全是字符串！total是'2598'不是2598

读到500行
{'order_id': 'O1000', 'customer_id': 'C007', 'product': 'Keyboard', 'category': 'Accessory', 'quantity': '2', 'price': '1299', 'order_date': '2024-01-01', 'country': 'Germany', 'total': '2598'}


In [12]:
# 写CSV——DictWriter
output = [
    {"name": "Alice", "score": 90},
    {"name": "Bob", "score": 85},
]
with open("score.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score"])
    writer.writeheader()
    writer.writerows(output)
# Windows上写CSV必须加newline=""，否则每行之间多一个空行

In [13]:
import json

# CSV → JSON：读csv，处理，写json（数据岗常见小任务）
summary = {"total_rows": len(rows), "sample": rows[0]}
with open("summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
    # ensure_ascii=False让中文正常显示；indent=2美化缩进

两个Windows专属坑记牢：写CSV加newline=""（否则空行），读写都加encoding="utf-8"（否则中文乱码）。